<a href="https://colab.research.google.com/github/thiagolimaop/AB-Testing-HomePage/blob/main/A_B_Testing_Home_Page.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A/B Testing Home Page

Imagine you work for a fictional productivity software company aiming to boost the number of paying customers. Currently, the software offers a free 7-day trial period during which users can download and use it without charge. Once the trial ends, users must purchase a license to keep using the software.


The company is considering redesigning the homepage to make the availability of a 7-day free trial more prominent and visible higher up on the page. The concern is that some potential users may not be aware of the trial and therefore miss the opportunity to experience the software. By increasing awareness of the trial, the company hopes to encourage more users to download and try the software, ultimately increasing the likelihood that they will purchase it after experiencing its features and value.

The company's website has five main sections:

1. the homepage;
2. a section with additional information, gallery, and examples;
3. a page for users to download the software;
4. a page for users to purchase a license; and
5. a support sub-site with documentation and FAQs for the software.

To download the software, users are required to create an account on the website. Once downloaded, the software can be used free of charge for seven days. After the trial period expires, the program displays a dialog box directing the user to the license page, where they can purchase a license to continue using the software.

## Metrics to track

In terms of metrics, it may be useful to track the number of cookies recorded at different stages of the website journey, particularly on the homepage, download page, and account registration page, which users must complete before downloading the software. These metrics can help us understand how users move through the conversion funnel.

Rather than analyzing only absolute counts, we could also examine ratios between these metrics. For example, we could calculate the proportion of homepage visitors who proceed to download the software. Similarly, license purchases could be expressed as a percentage of the initial number of homepage visitors, allowing us to measure the overall conversion rate.

## Other informations

- Unique visits per day: 3250
- Downloads per day: 520
- Licenses purchased each day: 65

Consider we are looking an improvement of 50 downloads per day and an increase of 10 licences purchased per day.

# Experiment Sizing

The first step is to calculate the experiment size for each group.
Considering that we have two evoluation metrics:

1. Downloads rate;
2. Licences purchased rate;

We will use Bonferroni correction to run each experiment. Since the type I error rate is 5%, use Bonferroni correction we will have 2.5% for  type I error rate.

Let's use statistical power as 80%, and define on-sided test to find the experiment size for each group. This way we have:

- H0: rate_control = rate_experiment
- H1: rate_control < rate_experiment

In [ ]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

p_a = 520/3250
p_b = 570/3250

effect_size = proportion_effectsize(p_b, p_a)

analysis = NormalIndPower()

n = analysis.solve_power(
    effect_size=effect_size,
    alpha=0.025,
    power=0.80,
    ratio=1,
    alternative="larger"
)

n_days = n * 2 / 3250
print(f"Number of days: {n_days:.0f}")

Number of days: 6


In [ ]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

p_a = 65/3250
p_b = 75/3250

effect_size = proportion_effectsize(p_b, p_a)

analysis = NormalIndPower()

n = analysis.solve_power(
    effect_size=effect_size,
    alpha=0.025,
    power=0.80,
    ratio=1,
    alternative="larger"
)

n_days = n * 2 / 3250
print(f"Number of days: {n_days:.0f}")

Number of days: 21


We should require 6 days to reliably detect a 50 download increase per day and 21 days to detect an increase of 10 license purchases per day.

In [2]:
import pandas as pd
import numpy as np
import scipy.stats as stats

url = 'https://raw.githubusercontent.com/thiagolimaop/AB-Testing-HomePage/5ad0954e84630ac406411f4d8dcff78ed33c6d8c/homepage-experiment-data.csv'
df = pd.read_csv(url)
df.tail()

,Day,Control Cookies,Control Downloads,Control Licenses,Experiment Cookies,Experiment Downloads,Experiment Licenses
24,25,1529,226,34,1609,284,29
25,26,1612,276,36,1574,295,38
26,27,1760,291,42,1666,297,28
27,28,1548,271,38,1587,314,37
28,29,1781,254,33,1728,290,41


# Invariant Metric

First, we should verify that the number of visitors assigned to each group is approximately balanced. Checking invariant metrics is an important prerequisite because it ensures that our conclusions about the evaluation metrics are based on comparable groups. If we find a statistically significant imbalance between the groups, we should investigate the assignment process to identify potential sources of bias. In some cases, this may indicate that the randomization procedure needs to be revised and the data collection repeated.

In this case, we would use a two-sided hypothesis test to determine whether the proportion of visitors assigned to one condition differs significantly from the expected proportion.

In [ ]:
n_control = df['Control Cookies'].sum()
n_total = df['Control Cookies'].sum() + df['Experiment Cookies'].sum()

p = 0.5
sd = np.sqrt(p * (1-p) * n_total)
z = ((n_control + 0.5) - p * n_total) / sd

p_value = 2 * stats.norm.cdf(z)
print(f"Z-score: {z}")
print(f"p-value: {p_value}")

Z-score: -1.6095646049678511
p-value: 0.10749294050130412


Since we lack sufficient reason to reject the null, we can continue on to evaluating the evaluation metrics.

# Evaluation Metrics

The download rate is the total number of downloads divided by the number of cookies, and the license purchasing rate the number of licenses divided by the number of cookies.

One tricky point to consider is that there is a seven or eight day delay between when most people download the software and when they make a purchase. There's no direct way of attributing cookies all the way through license purchases due to the daily aggregation of results, so the best we can do is to make a justified argument for handling the data. About the license purchasing rate, we should only take the cookies observed through day 21 as the denominator of the ratio as being responsible for all of the license purchases observed.

We need to detect an increase of 10 license purchases per day, and detect a 50 download increase per day.

In [ ]:
n_experiment_licence = df[df['Day'] <= 21]['Experiment Cookies'].sum()
n_control_licence = df[df['Day'] <= 21]['Control Cookies'].sum()

licence_rate_c = df['Control Licenses'].sum() / n_control_licence * 100
licence_rate_e = df['Experiment Licenses'].sum() / n_experiment_licence * 100

print(f"Control Licence Rate: {licence_rate_c:.2f}%")
print(f"Experiment Licence Rate: {licence_rate_e:.2f}%")

Control Licence Rate: 2.10%
Experiment Licence Rate: 2.13%


In [ ]:
n_experiment_download = df['Experiment Cookies'].sum()
n_control_download = df['Control Cookies'].sum()

download_rate_c = df['Control Downloads'].sum() / n_control_download * 100
download_rate_e = df['Experiment Downloads'].sum() / n_experiment_download * 100

print(f"Control Download Rate: {download_rate_c:.2f}%")
print(f"Experiment Download Rate: {download_rate_e:.2f}%")

Control Download Rate: 16.12%
Experiment Download Rate: 18.05%


Knowing that we want to detect 10 licence purchases per day, what is the rate we want achieve, i.e, MDE?

In [ ]:
licences_per_day_c = (n_control_licence * licence_rate_c / 100 / 21)
licences_per_day_exp = licences_per_day_c + 10
licence_rate_exp = (licences_per_day_exp * 21 / n_control_licence) * 100

print(f"Control Licence Rate: {licence_rate_c:.2f}%")
print(f"Expected Licence Rate: {licence_rate_exp:.2f}%")
print(f"MDE: {licence_rate_exp - licence_rate_c:.2f}%")

Control Licence Rate: 2.10%
Expected Licence Rate: 2.73%
MDE: 0.62%


Knowing that we want to detect 50 download increase per day, what is the rate we want achieve, i.e, MDE?

In [ ]:
download_per_day_c = (n_control_download * download_rate_c / 100 / 29)
download_per_day_exp = download_per_day_c + 50
download_rate_exp = (download_per_day_exp * 29 / n_control_download) * 100

print(f"Control download Rate: {download_rate_c:.2f}%")
print(f"Expected download Rate: {download_rate_exp:.2f}%")
print(f"MDE: {download_rate_exp - download_rate_c:.2f}%")

Control download Rate: 16.12%
Expected download Rate: 19.22%
MDE: 3.09%


Now, it's possible to calculate the p_value for both metrics.

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

conversions = [df['Experiment Licenses'].sum(), df['Control Licenses'].sum()]
sample_sizes = [n_experiment_licence, n_control_licence]

z_stat, p_value = proportions_ztest(
    count=conversions,
    nobs=sample_sizes,
    alternative="larger"
)

print(f"Z = {z_stat:.2f}")
print(f"p-value = {p_value:.4f}")

Z = 0.26
p-value = 0.3979


In [ ]:
from statsmodels.stats.proportion import proportions_ztest

conversions = [df['Experiment Downloads'].sum(), df['Control Downloads'].sum()]
sample_sizes = [n_experiment_download, n_control_download]

z_stat, p_value = proportions_ztest(
    count=conversions,
    nobs=sample_sizes,
    alternative="larger"
)

print(f"Z = {z_stat:.2f}")
print(f"p-value = {p_value:.4f}")

Z = 7.87
p-value = 0.0000


# Conclusions

We reject the hypotheses that Experiment downloads are equal Control downloads, once the p-value is really small, and fail to reject the hypotheses that Experiment Licences are equal to Control licences since p-value in this case is greater than 0.025!

We can go ahead with the deploy, since the objective was achieved, i.e, one of the two tests demonstrated statistical significance.